In [ ]:
import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
# tv vs tv + drift, expecting scatter barely above the unity and a wider distro along the x-axis

In [ ]:
from core.data import load_sess, get_encoder_io

(
    spike_times,
    trial_data,
    psths,
    session_data,
    regions,
) = load_sess(
    subj_id=subj_id,
    sess_id=sess_id,
    tpre=0.5,
    tpost=1,
    alignment="choice",
    binwidth_ms=25,
    add_svd=False,
    thresh=1,
)

(
    tents,
    tvs,
    dm,
    robs,
    dm_names,
    reg_idxs,
) = get_encoder_io(
    psths,
    trial_data,
    regions,
    norm=False,
    num_tents=5,
    tv_keys=["response", "rewarded", "block_side", "response_prev", "rewarded_prev"],
    add_svd=False,
    num_svd=None,
    binwidth_ms=25,
)

num_trials, num_tv = tvs.shape
num_units = robs.shape[1]

In [ ]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score


def get_scores(dm, robs, train_idxs, test_idxs):
    drift_model = RidgeCV(
        alphas=np.logspace(-5, 5, 11, base=10), alpha_per_target=True
    ).fit(dm[train_idxs], robs[train_idxs])

    scores = r2_score(
        robs[test_idxs],
        drift_model.predict(dm[test_idxs]),
        multioutput="raw_values",
        force_finite=False,
    )

    return scores

In [ ]:
n_folds = 10
p_train = 0.8

scores_cv = {
    "drift": np.zeros((n_folds, num_units)),
    "tv": np.zeros((n_folds, num_units)),
    "encoder": np.zeros((n_folds, num_units)),
}

for i in range(n_folds):
    np.random.seed(seed=i)
    train_idxs = np.sort(
        np.random.choice(num_trials, int(num_trials * p_train), replace=False)
    )
    test_idxs = np.setdiff1d(np.arange(num_trials), train_idxs)

    scores_cv["drift"][i] = get_scores(
        dm=tents, robs=robs, train_idxs=train_idxs, test_idxs=test_idxs
    )
    scores_cv["tv"][i] = get_scores(
        dm=tvs, robs=robs, train_idxs=train_idxs, test_idxs=test_idxs
    )
    scores_cv["encoder"][i] = get_scores(
        dm=np.hstack((tents, tvs)),
        robs=robs,
        train_idxs=train_idxs,
        test_idxs=test_idxs,
    )

scores = {
    "drift": np.median(scores_cv["drift"], axis=0),
    "tv": np.median(scores_cv["tv"], axis=0),
    "encoder": np.median(scores_cv["encoder"], axis=0),
}

In [ ]:
keys = ["a", "b", "c"]
for bleb in zip(keys, keys[1:]):
    print(bleb)

In [ ]:
test = {
    "a": np.array([10, 0, -10]),
    "b": np.array([0, -10, 10]),
    "c": np.array([-10, 10, 1]),
}

In [ ]:
comparisons = [test[a] > test[b] for a, b in zip(keys, keys[1:])]
comparisons
# np.mean(np.all(comparisons, axis=0)).round(3)

In [ ]:
np.all(comparisons, axis=0)

In [ ]:
from itertools import permutations, product


def p_chain(keys):
    comparisons = [scores[a] > scores[b] for (a, b) in zip(keys, keys[1:])]
    return np.mean(np.all(comparisons, axis=0)).round(3)


models_permutate = permutations(scores.keys())
p_trio = {"_".join(models): p_chain(models) for models in models_permutate}

models_pair = product(scores.keys(), scores.keys())
p_pair = {
    "_".join(models): p_chain(models)
    for models in models_pair
    if not models[0] == models[1]
}

In [ ]:
assert np.isclose(np.sum([p_trio[models] for models in p_trio]), 1, 1e-3)

In [ ]:
from core.utils import pprint_flat

pprint_flat(p_pair)

In [ ]:
(
    np.mean(scores["tv"]).round(3),
    np.mean(scores["drift"]).round(3),
    np.mean(scores["encoder"]).round(3),
)

In [ ]:
def style_legend(legend, colors):
    legend.get_frame().set_visible(False)
    for text, color in zip(legend.get_texts(), colors):
        text.set_color(color)
        text.set_fontfamily("sans-serif")
        text.set_fontweight("bold")

In [ ]:
fig, ax = plt.subplots()

ax.scatter(
    scores["drift"], scores["encoder"], s=0.5, color="#0F4676", label="encoder"
)  # 66 is a unit close to the unity line
ax.plot([0, 1], color="#666666", linestyle="--", linewidth=0.5)
ax.axhline(y=0, color="#222222", linewidth=0.5)
ax.axvline(x=0, color="#222222", linewidth=0.5)
ax.set_zorder(2)
ax.patch.set_visible(False)

ax.set_ylim([-0.2, 1.1])
ax.set_xlabel("drift")
ax.set_ylabel("encoder")

ax2 = ax.twinx()
ax2.scatter(
    scores["drift"], scores["tv"], s=0.5, color="#7BA1D2", label="tv"
)  # 66 is a unit close to the unity line
ax2.set_ylim([-0.2, 1.1])
ax2.set_ylabel("tv")
ax2.set_zorder(1)


legend = fig.legend(markerscale=0, bbox_to_anchor=(0.8, 0.95))
style_legend(legend, ["#0F4676", "#7BA1D2"])

fig.tight_layout()

In [ ]:
0.168 + 0.298 + 0.405  # encoder > tv

In [ ]:
0.092 + 0.298 + 0.405  # encoder > drift

In [ ]:
A = np.hstack(
    (
        scores["drift"].reshape(-1, 1),
        scores["encoder"].reshape(-1, 1),
        scores["tv"].reshape(-1, 1),
    )
)

plt.figure()
plt.imshow(A, aspect="auto")
plt.show()